## NLP Final
### Part 4: Train Sentiment Model
### Aren Mizuno
### March 12, 2026

In [ ]:
# Imports
!pip -q install -U transformers accelerate datasets evaluate scikit-learn pyarrow
import os
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from google.colab import drive

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 4.7 MB/s eta 0:00:00


In [ ]:
# Cuda
torch.cuda.is_available()

False

In [ ]:
# Mount drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# Load dataset
ds = load_dataset("sara-nabhani/ML-news-sentiment")
ds.head(5)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


sentiment_train.csv: 0.00B [00:00, ?B/s]

sentiment_test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/4551 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/506 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'label_name'],
        num_rows: 4551
    })
    test: Dataset({
        features: ['text', 'labels', 'label_name'],
        num_rows: 506
    })
})

In [ ]:
# Check train
print(ds["train"].to_pandas().head(5))

                                                text  labels label_name
0  Finnish airline Finnair is starting the tempor...       0   negative
1  The corresponding increase in the share capita...       1    neutral
2  In the third quarter of fiscal 2008 Efore swun...       0   negative
3  ALEXANDRIA , Va. , Oct. 15 -- Aaron Moss of Ha...       1    neutral
4  Vaisala Oyj Stock exchange release 26.03.2010 ...       1    neutral


In [ ]:
# Check test
print(ds["test"].to_pandas().head(5))

                                                text  labels label_name
0  COPYRIGHT AFX News and AFX Financial News Logo...       1    neutral
1  Satama and Trainers ' House will remain as nam...       1    neutral
2  Prices and delivery volumes of broadband produ...       0   negative
3  When open next year , it will be the largest f...       2   positive
4  The tower 's engineers have created an 18 degr...       1    neutral


In [ ]:
# Rename column to match Trainer expectations
ds = ds.rename_column("labels", "label")

In [ ]:
# Keep only columns needed
ds["train"] = ds["train"].remove_columns(["label_name"])
ds["test"]  = ds["test"].remove_columns(["label_name"])

In [ ]:
# Tokenizer
model_name = "distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True)

ds_train_tok = ds["train"].map(tokenize, batched=True)
ds_test_tok  = ds["test"].map(tokenize, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/4551 [00:00<?, ? examples/s]

Map:   0%|          | 0/506 [00:00<?, ? examples/s]

In [ ]:
# Model
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

In [ ]:
# Training arguments
args = TrainingArguments(
    output_dir="/content/my_sentiment_model_runs",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    report_to="none",
)

In [ ]:
# Trainer
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_train_tok,
    eval_dataset=ds_test_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
# Train
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,No log,0.319911,0.861660,0.861591
2,0.450147,0.322276,0.859684,0.861149
3,0.450147,0.342565,0.869565,0.870506


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=855, training_loss=0.35920589513946, metrics={'train_runtime': 4105.7144, 'train_samples_per_second': 3.325, 'train_steps_per_second': 0.208, 'total_flos': 206906900059404.0, 'train_loss': 0.35920589513946, 'epoch': 3.0})

In [ ]:
# Evaluate
metrics = trainer.evaluate()
print(metrics)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 0.3425650894641876, 'eval_accuracy': 0.8695652173913043, 'eval_f1_macro': 0.8705056954218965, 'eval_runtime': 45.8486, 'eval_samples_per_second': 11.036, 'eval_steps_per_second': 0.349, 'epoch': 3.0}


In [ ]:
# Classification report
pred = trainer.predict(ds_test_tok)

test_preds = np.argmax(pred.predictions, axis=-1)
test_labels = np.array(ds_test_tok["label"])

print(
    classification_report(
        test_labels,
        test_preds,
        target_names=["negative", "neutral", "positive"]
    )
)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


              precision    recall  f1-score   support

    negative       0.87      0.92      0.89        87
     neutral       0.90      0.86      0.88       268
    positive       0.82      0.85      0.84       151

    accuracy                           0.87       506
   macro avg       0.86      0.88      0.87       506
weighted avg       0.87      0.87      0.87       506



In [ ]:
# Save models
drive.mount("/content/drive")

save_dir = "/content/drive/MyDrive/UChicago/Masters/Winter/NLP/Final Project/sentiment_model"

trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

print("Saved model+tokenizer to:", save_dir)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model+tokenizer to: /content/drive/MyDrive/UChicago/Masters/Winter/NLP/Final Project/sentiment_model
